In [1]:
!pip install langchain langchain-community sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.

In [9]:
with open("my_info.txt", "w") as f:
    f.write("""My name is Nagib. I am a CSE graduate learning AI Engineering.
I started this journey on February, 2026.
My favorite programming language is Python.
I completed a machine learning project using the Titanic dataset.
I built a CLI chatbot using the Groq API.""")

In [5]:
!pip install langchain-text-splitters

In [16]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:0000:01


In [20]:
from kaggle_secrets import UserSecretsClient
from groq import Groq

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("groq_api_key")

client = Groq(api_key=api_key)

In [10]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("my_info.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")
for chunk in chunks:
    print(chunk.page_content)
    print("---")

Total chunks: 4
My name is Nagib. I am a CSE graduate learning AI Engineering.
---
I started this journey on February, 2026.
My favorite programming language is Python.
---
I completed a machine learning project using the Titanic dataset.
---
I built a CLI chatbot using the Groq API.
---


In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

query = "What programming language do I like?"
results = vectorstore.similarity_search(query, k=2)
for r in results:
    print(r.page_content)

/tmp/ipykernel_58/763342152.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

I started this journey on February, 2026.
My favorite programming language is Python.
My name is Nagib. I am a CSE graduate learning AI Engineering.


In [21]:
def rag_query(question):
    relevant_chunks = vectorstore.similarity_search(question, k=2)
    context = "\n".join([chunk.page_content for chunk in relevant_chunks])
    
    prompt = f"""Answer the question based only on the following context:

Context:
{context}

Question: {question}

Answer:"""
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

print(rag_query("What did I build using the Groq API?"))
print(rag_query("When did I start this journey?"))

A CLI chatbot.
February 2026.


**RAG কে PDF এর উপর কাজ করানো + Streamlit দিয়ে Web App বানানো**

In [22]:
!pip install pypdf

In [29]:
import os
for dirname, _, filenames in os.walk('/kaggle/input/datasets/mdnagibmahfuj/bangladesh'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/mdnagibmahfuj/bangladesh/BangladeshsCurrentSituation-ProblemsandSolut....pdf


In [30]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/kaggle/input/datasets/mdnagibmahfuj/bangladesh/BangladeshsCurrentSituation-ProblemsandSolut....pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

vectorstore = FAISS.from_documents(chunks, embeddings)

In [34]:
def rag_query(question):
    relevant_chunks = vectorstore.similarity_search(question, k=2)
    context = "\n".join([chunk.page_content for chunk in relevant_chunks])
    
    prompt = f"""Answer the question based only on the following context:

Context:
{context}

Question: {question}

Answer:"""
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [35]:
print(rag_query("এই PDF এ কী নিয়ে লেখা?"))

সরকার
 
অবহিত
 
করিয়াছে
 
যেম
 
চলতি
 
বছর
 
কালের 
 
শেষের
 
দিকে
 
স্বল্পমেয়াদি
 
ঋণ
 
প্রদান
 
বন্ধ
 
করে
 
দেওয়া
 
হবে
 
৷
 
সরকার
 
স্বল্পমেয়াদ
 
অর্থায়ন
 
প্রদান
 
বন্ধ
 
করে
 
দেয়ার
 
সিদ্ধান্ত 
 
নিয়েছে
 
মুদ্রাস্ফীতি
 
রােধ
 
এবং
 
চলতি
 
অবস্থার 
 
উন্নয়ন
 
প্রতিরােধ
 
করতে
 
৷
 
সরকার
 
নির্বাচন
 
করেছে
 
স্বল্পমেয়াদি
 
ঋণ
 
প্রদান
 
বন্ধ
 
করে
 
দেয়ার
 
পথ
 
মুদ্রাস্ফীতি
 
কমাতে
 
৷
 
বাংলাদেশ
 
ব্যাংক
 
নােট
 
বর্তমানে
 
কমপক্ষ
 
৫
 
লাখ
 
কোটি
 
টাকা
 
মূল্যের
 
স্বল্পমেয়াদি
 
ঋণ
 
প্রদান
 
করছে
 
৷
 
যদি 
 
মুদ্রাস্ফীতি 
 
কমে
 
যায়
 
স্বল্পমেয়াদ
 
অর্থায়ন 
 
নিরাপদ
 
হবে
 
এবং
 
তারপর
 
স্বল্পমেয়াদ
 
অর্থায়ন
 
প্রদান
 
চালু 
 
করা
 
হবে
 
৷
 
এটা
 
স্পষ্ট
 
যে
 
সরকার
 
স্বল্পমেয়াদি
 
ঋণ
 
প্রদান
 
বন্ধ
 
করার
 
সিদ্ধান্ত
 
নিয়েছে
 
অস্থায়ী
 
সময়ের
 
জন্য
 
৷
 
ব্যাংক
 
স্বল্পমেয়াদ
 
অর্থায়ন
 
প্রদান 
 
করে
 
থাকে
 
এভাবে
 
মুদ্রাস্ফীতি
 
বৃদ্ধি
 
পাচ্ছে
 
সেই
 
ব্যাংক
 
স্বল্পমেয়াদ
 
অর্থায়ন
 
প্রদান
 
বন্ধ
 
করে
 
দেবে
 
৷
 
স্বল্পমেয়াদি
 
ঋণ
 
প্রদান
 
বন্ধ
 
করা 
 


In [39]:
print(rag_query("give me short summery"))

া (ভৌগোলিক অবস্থান)
 
- 
 
জলবায়ু-পরিবর্তনের প্রভাবে উদ্ভূত চ্যালেঞ্জ মোকাবেলায় 
আন্তর্জাতিক বিশ্ববিদ্যালয়ের মাঠপর্যায়ের কার্যক্রমের মধ্যে একত্রে বাংলাদেশের সাথে কলকাতার আন্তর্জাতিক বিশ্ববিদ্যালয়সমূহের একটি সংস্থা রয়েছে।
 
আন্তর্জাতিক পরিবেশ অধ্যয়ণ ও ব্যবস্থাপনা ইনস্টিটিউট (আইইইএম)-এর তথ্য অনুযায়ী বাংলাদেশ এবং কলকাতার দুই আন্তর্জাতিক বিশ্ববিদ্যালয়ের একটি সমন্বিত উদ্যোগ রয়েছে 'সচেতনতা বৃদ্ধি ও জলবায়ু পরিবর্তন মোকাবেলা' নামক প্রকল্পের অন্তর্গত।
 
কলকাতা ও বাংলাদেশের মুখোমুখি হওয়া পরিবেশগত বিষয়গুলোর মধ্যে জীবাশ্ম জ্বালানি নির্ভরতা, দূষিত পানি বর্জ্য এবং আবর্জনা ব্যবস্থাপনা রয়েছে।
 
'২০০৯' থেকে '২০১০' সাল পর্যন্ত এই প্রকল্প চলবে বলে ধারণা করা হয়েছিল।
 
প্রকল্পটিতে বাংলাদেশ ও কলকাতার জলবায়ু, পিরবতা (ভৌগোলিক অবস্থান) এবং জলবায়ু-পরিবর্তনের প্রভাবে উদ্ভূত চ্যালেঞ্জ মোকাবেলায় তথ্য সংগ্রহ করা হবে।
 
বাংলাদেশের যে সকল জায়গায় এই প্রকল্প পরিচালনা করা হবে সেই জায়গাগুলো হলো ঢাকা, চট্টগ্রাম ও খুলনা।
 
ঢাকা, চট্টগ্রাম ও খুলনার সমষ্টিগত জনগণের সচেতনতা বৃদ্ধির উদ্দেশ্যে বাংলাদেশে এই প্রকল্প গ্রহণ করা হয়ে

**Streamlit ইনস্টল ও বেসিক অ্যাপ**